In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from glob import glob
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import Javascript, display
from IPython.display import HTML, Image
from pathlib import Path

In [ ]:
def read_results(path):
    #print(f"\tTrying to read {path}")
    if os.path.exists(path):
        with open(path) as f:
            results = json.load(f)
        return results
    else:
        print(f"Cannot find file {path}")
        return None

lang_map = {"eng": "en",
            "fra": "fr",
            "fin": "fi",
            "zho": "zh",
            "ita": "it",
            "deu": "de",
            "vie": "vi",
            "jpn": "ja",
            "ukr": "uk",
            "ell": "el",
            "hin": "hi",
            "bul": "bg",
            "fas": "fa"}

In [ ]:
models = [os.path.basename(m) for m in glob("path_to_models")] + ["openai__text-embedding-3-small","openai__text-embedding-3-large", "gemini-embedding-001"]


dataset_map ={"ARCChallenge" : "arcchallenge",
              "RTE3-multi": "rte3-multi__contradiction",
              "Tatoeba": "tatoeba",
              "SummEvalSummarization.v2": "summeval-2",
              "WebFAQRetrieval": "web-faq-bitext"}

In [ ]:
def parse_mteb_results(models, dataset):
    if dataset == "RTE3-multi":
        return parse_rte(models, dataset)
    print("Reading MTEB results...")
    # path where the results are
    mteb_results_path=lambda x: [i for i in glob(f"mteb_out/{x}/results/{x}/*/{dataset}.json")]
    results = {}
    for model in models:
        #print(f"In {model}")
        if model == "gemini-embedding-001":
            model_to_read = "google__gemini-embedding-001"
        else:
            model_to_read = model
        mteb_results_path_ = mteb_results_path(model_to_read)
        if mteb_results_path_ == []:
            print(f"\tModel {model} missing results from mteb")
            continue
        if len(mteb_results_path_) != 1:
            # the dataset revision has changed, using the first results
            print(f"\t{len(mteb_results_path_)} evaluation files found for {model}. Selecting first results.")
            mteb_results_path_ = [mteb_results_path_[0]]
        
        d = read_results(mteb_results_path_[0])  # [0] just to remove nesting
        for subsplit in d["scores"]["test"]:
            #print(subsplit)
            if not f'{subsplit["hf_subset"]}' in results.keys():
                results[f'{subsplit["hf_subset"]}'] = {}
            results[f'{subsplit["hf_subset"]}'][f'{model}']= subsplit["main_score"]
    return results

def parse_rte(models, dataset):
    results = {}
    langs_rte = ["en", "fr", "it", "de"]
    mteb_results_path="mteb_out/rte_crawled.json"
    j = read_results(mteb_results_path)
    for m in models:
        m_ = m.replace("__", "/")
        if "gemini" in m_:
            m_ = "google/gemini-embedding-001"
        for l in langs_rte:
            if l not in results.keys():
                results[l] = {}
            #print(f"{m_}_{l}")
            results[l][m] = j[f"{m_}_{l}"]
    return results

            



In [ ]:
def parse_bss_results(models, dataset, splits, prompt=False, shuffled=False, method="ICA", dim=32):
    print("Reading BSS results...")
    ginis = {}
    peaks = {}
    peak_indices = {}
     # unneccessarily complex path
    bss_results_path=lambda x: f"stats{'_with_prompt' if prompt else ''}/{method}_{dim}/{dataset}/{x}.json" if isinstance(x, str) else f"stats{'_with_prompt' if prompt else ''}/{method}_{dim}/{dataset}:{x[1]}/{x[0]}.json"
   

    for m in models:
        if "openai" in m:
            m_ = m.replace("openai__","")
        else:
            m_ = m

        for s in splits:
            if s == "default":
                path = bss_results_path(m_)
            else:
                if dataset == "tatoeba":
                    # remap splits to match hf format
                    s_ = lang_map[s.split("-")[1]]+"-"+lang_map[s.split("-")[0]]
                    path = bss_results_path([m_, s_])
                else:
                    path = bss_results_path([m_, s])
            if not os.path.isfile(path):
                print(f"\tmissing results for {path}")
                continue
            else:
                j = read_results(path)
            if s not in ginis.keys():
                ginis[s] = {}
                peaks[s] = {}
                peak_indices[s] = {}
            ginis[s][m] = (dim-1)/dim - j["gini"]  # see analysis/README.md
            peaks[s][m] = j["peak"]
            peak_indices[s][m] = j["peak_indices"]
        
    return ginis, peaks, peak_indices




In [ ]:
def parse_NN_results(models, dataset, splits, prompt=False):
    print("Reading NN-results...")
    NN_results_path= lambda model, data: f"linearity_results{'_with_prompt' if prompt else ''}/{data}/{model}.json"   #old results in linearity_results_cosine_recall
    
    knns = {}
    r2s = {}
    for m in models:
        if "openai" in m:
            m_ = m.replace("openai__","")
        else:
            m_ = m
        for s in splits:
            if s == "default":
                path = NN_results_path(m_, dataset)
            else:
                if dataset == "tatoeba":
                    # remap splits to match hf format
                    s_ = lang_map[s.split("-")[1]]+"-"+lang_map[s.split("-")[0]]
                    path = NN_results_path(m_, dataset+":"+s_)
                else:
                    path = NN_results_path(m_, dataset+":"+s)
            if not os.path.isfile(path):
                print(f"\tmissing results for {path}")
                continue
            else:
                j = read_results(path)
                #print(j)
            if s not in knns.keys():
                knns[s] = {}
                r2s[s] = {}
            if dataset == "SummEvalSummarization.v2":
                knns[s][m] = j["knn_scores"]["5"]
                r2s[s][m] = j["r2_affine"]
            else:
                knns[s][m] = j["knn_scores_percentage"]["10%"]
                r2s[s][m] = j["r2_affine"]
           
    return knns, r2s



In [ ]:
def color_text(s):
    return f"\x1b[31m{s}\x1b[0m"

def analysis(dataset, df, score, against,split="default", pprint=False):
    collect_results = []
    if isinstance(score, str):
        score = [score]
    for s in score:
        x = df[s].tolist()
        y = df[against].tolist()
        pr = pearsonr(x,y)
        sp = spearmanr(x,y)
        p_value = round(pr.pvalue,5)
        p_value = color_text(p_value) if p_value <= 0.05 else p_value
        sp_value = round(sp.pvalue, 5)
        sp_value = color_text(sp_value) if sp_value <= 0.05 else sp_value
        if pprint:
            print(f"\t Statistic: {s} (support {len(x)})\t\tP-Result: {round(pr.statistic, 3)} (p-value {p_value})  \n\
                \t\t\t\tS-Result: {round(sp.statistic, 3)} (p-value {sp_value})")
        collect_results.append({"dataset": dataset, "split":split, "statistic":s, "support": len(x), "pr": (pr.statistic, pr.pvalue), "sp":(sp.statistic, sp.pvalue)})
    return collect_results
        



def make_table_old(dataset, ginis32, ginis64, ginis128, peaks32, peaks64, peaks128, knns, r2s, mteb_scores):
    results_knn = []
    results_bss = []
    for split in mteb_scores.keys():
        #print(mteb_scores[split])
        df_mteb = pd.DataFrame([mteb_scores[split]]).T.rename(columns={0:"mteb_score"})
        df_ginis32 =pd.DataFrame([ginis32[split]]).T.rename(columns={0:"gini32"})
        df_ginis64 =pd.DataFrame([ginis64[split]]).T.rename(columns={0:"gini64"})
        df_peaks32 =pd.DataFrame([peaks32[split]]).T.rename(columns={0:"peak32"})
        df_peaks64 =pd.DataFrame([peaks64[split]]).T.rename(columns={0:"peak64"})
        df_knn = pd.DataFrame([knns[split]]).T.rename(columns={0:"knn_avg"})
        df_r2s = pd.DataFrame([r2s[split]]).T.rename(columns={0:"r2_aff"})
        df = pd.concat([df_mteb, df_ginis32, df_ginis64, df_peaks32, df_peaks64, df_knn, df_r2s], axis=1)
        df = df.reset_index().rename(columns={"index":"model"})
        df = df.dropna()
        #display(df.head())
        #print(split)
        r1 = analysis(dataset, df, ["knn_avg", "r2_aff"], "mteb_score", split=split)
        r2 = analysis(dataset, df, ["gini32", "gini64", "peak32", "peak64"], "mteb_score", split=split)
        results_knn.append(r1)
        results_bss.append(r2)
    return results_knn, results_bss


def make_table(dataset, mteb_scores, dict_of_results):
    results_knn = []
    results_bss = []
    for split in mteb_scores.keys():
        dfs = []
        df_mteb = pd.DataFrame([mteb_scores[split]]).T.rename(columns={0:"mteb_score"})
        for name, result in dict_of_results.items():
            df = pd.DataFrame([result[split]]).T.rename(columns={0:name})
            dfs.append(df)
        df = pd.concat([df_mteb] + dfs,  axis=1)
        df = df.reset_index().rename(columns={"index":"model"})
        df = df.dropna()   # drops all NaN values --> i.e. do not analyse unless all results exist
        r1 = analysis(dataset, df, [i for i in dict_of_results.keys() if i in ["knn_avg", "r2_aff"]], "mteb_score", split=split)
        r2 = analysis(dataset, df, [i for i in dict_of_results.keys() if "gini" in i or "peak" in i], "mteb_score", split=split)
        results_knn.append(r1)
        results_bss.append(r2)
    return results_knn, results_bss





In [ ]:

def format_corr_value(corr_tuple: tuple[float, float]) -> str:
    """Format a (coefficient, pvalue) tuple into a LaTeX string."""
    coeff, pvalue = corr_tuple

    coeff_str = f"{coeff:.3f}"

    # Significance marker based on p-value
    if pvalue > 0.05:
        sig = ""
    elif pvalue > 0.01:
        sig = r"$^{*}$"
    elif pvalue > 0.001:
        sig = r"$^{\dagger}$"
    else:
        sig = r"$^{\ddagger}$"

    # Bold if |coeff| >= 0.7
    if abs(coeff) >= 0.7:
        return r"\textbf{" + coeff_str + "}" + sig
    else:
        return coeff_str + sig


def to_latex_rows_old_with_no_prompt(results, statistics_to_include="all", metrics_to_include=["spearman", "spearman_wp"]):
    prev_dataset=None
    rows = []
    prev_dataset = None
    val_map = {"knn_avg": "$k$NN", "r2_aff": "$r2$"}
    for data in results:
        for row in data:
            dataset   = row["dataset"]
            split = row["split"]
            statistic = row["statistic"]
            statistic = row["statistic"]
            if statistics_to_include != "all" and statistic not in statistics_to_include and statistic != statistics_to_include:
                print(f"not including {statistic=}")
                continue
            pearson   = format_corr_value(row["pr"])
            spearman  = format_corr_value(row["sp"])


            rows.append(f"{dataset}:{split if split!='default' else ''} & {val_map[statistic]} '&'.join{pearson} & {spearman} \\\\")
            #prev_dataset = dataset+split


    return "\n".join(rows)

def to_latex_rows_with_prompt(results, results_with_prompt, statistics_to_include="all", metrics_to_include=["spearman", "spearman_wp"]):
    prev_dataset=None
    rows = []
    prev_dataset = None
    val_map = {"knn_avg": "$k$NN", "r2_aff": "$r2$"}
    dataset_name_map = {"WebFAQRetrieval": "WebFAQ", "SummEvalSummarization.v2": "SummEval"}
    for data, data_wp in zip(results, results_with_prompt):
        for row, row2 in zip(data, data_wp):
            dataset_name   = row["dataset"]
            split = row["split"]
            statistic = row["statistic"]
            if statistics_to_include != "all" and statistic not in statistics_to_include and statistic != statistics_to_include:
                #print(f"not including {statistic=}")
                continue
            d = {}
            d["pearson"]  = format_corr_value(row["pr"])
            d["spearman"]  = format_corr_value(row["sp"])
            d["pearson_wp"] = format_corr_value(row2["pr"])
            d["spearman_wp"] = format_corr_value(row2["sp"])

            if metrics_to_include == "all":
                metrics_to_include = ["pearson", "spearman", "pearson_wp", "spearman_wp"]
            m = " & ".join(d[v] for v in metrics_to_include)
            rows.append(f"{dataset_name_map.get(dataset_name, dataset_name)}{':'+split if split!='default' else ''}  & {m} \\\\")  # & {row['support']} #& {val_map.get(statistic, statistic)}  <- if you want this
            #prev_dataset = dataset_name+split
    return "\n".join(rows)
    

In [ ]:
def runner(models, dataset, dataset_):
    mteb_scores = parse_mteb_results(models, dataset)
    ginis32, peaks32, peak_indices32 = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=32)
    ginis64, peaks64, peak_indices64 = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=64)
    knns, r2s = parse_NN_results(models, dataset_, splits = [i for i in mteb_scores.keys()])

    ginis32_p, peaks32_p, peak_indices32_p = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=32, prompt=True)
    ginis64_p, peaks64_p, peak_indices64_p = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=64, prompt=True)
    knns_p, r2s_p = parse_NN_results(models, dataset_, splits = [i for i in mteb_scores.keys()], prompt=True)

    collected_results = {"ginis32": ginis32,  
                         "peaks32": peaks32,
                         "ginis64": ginis64, 
                         "peaks64": peaks64,
                         "knn_avg": knns, 
                         "r2_aff": r2s}
    collected_results_prompted = {"ginis32": ginis32_p,  
                         "peaks32": peaks32_p,
                         "ginis64": ginis64_p, 
                         "peaks64": peaks64_p,
                         "knn_avg": knns_p, 
                         "r2_aff": r2s_p}
    results_knn, results_bss = make_table(dataset, mteb_scores, collected_results)
    results_knn_p, results_bss_p = make_table(dataset, mteb_scores, collected_results_prompted)

    return results_knn, results_knn_p, results_bss, results_bss_p
    #return to_latex_rows_with_prompt(results_knn, results_knn_p, statistics_to_include=indicator)

In [ ]:
rows={"knn": [], "gini32": [], "gini64": []}
for d, d_ in dataset_map.items():
    print(d)
    results_knn, results_knn_p, results_bss, results_bss_p = runner(models, d,d_)
    #print(results_bss)
    rows["knn"].append(to_latex_rows_with_prompt(results_knn, results_knn_p, statistics_to_include="knn_avg"))
    rows["gini32"].append(to_latex_rows_with_prompt(results_bss, results_bss_p, statistics_to_include="ginis32"))
    rows["gini64"].append(to_latex_rows_with_prompt(results_bss, results_bss_p, statistics_to_include="ginis64"))



In [ ]:
def print_table(rows):
    prints = []
    current_row=0
    for i, r in enumerate(rows):
        if "SummEval" in r or "WebFAQ" in r:
            for r_ in r.split("\n"):
                prints[current_row] += r_
                current_row+=1
        else:
            #print("now in ", r)
            for r_ in r.split("\n"):
                prints.append(r_.replace("\\\\", " & & "))
    for r in prints:
        print(r)

print_table(rows["knn"])
#for r in rows["knn"]:
#    print("---")
#    print(r)

In [ ]:
print_table(rows["gini32"])
#print(rows.keys())

In [ ]:
print_table(rows["gini64"])